# Bab 2 — Internal Clustering Evaluation

Notebook pendamping Bab 2 Level 5.

Notebook ini mengikuti halaman bab di situs. Jalankan sel dari atas ke bawah.

## Learning Objectives

Setelah mempelajari bab ini, kamu mampu:

- Menjelaskan apa itu *internal evaluation* dan mengapa ia penting ketika tidak ada label benar.
- Menghitung *intra-cluster distance* dan *inter-cluster distance* secara manual dan dengan kode.
- Membedakan *compactness*, *cohesion*, *separation*, dan *density*.
- Menjelaskan hubungan antara jarak dan mutu kelompok.
- Membaca perbedaan skor pada kelompok yang dekat dan kelompok yang jauh.
- Menyebutkan kekuatan dan keterbatasan evaluasi internal, termasuk kasus ketika label benar justru mendapat skor lebih rendah.
- Menyiapkan fondasi untuk memahami Inertia, Silhouette, Davies-Bouldin, dan Calinski-Harabasz.

## Langkah 0: Pindah ke folder utama proyek

In [ ]:
import os
from pathlib import Path

if not Path("data").exists():
    os.chdir("../..")

## Bagian A: Siapkan alat

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import pairwise_distances, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

## Bagian B: Data kecil dan dua kelompoknya

In [ ]:
# Enam titik di dua kelompok yang sudah kita tentukan sendiri
X_tiny = np.array(
    [
        [1, 1], [2, 1], [1, 2],   # kelompok 0
        [6, 5], [7, 5], [6, 6],   # kelompok 1
    ],
    dtype=float,
)
labels_tiny = np.array([0, 0, 0, 1, 1, 1])

## Bagian C: Empat ukuran dasar sebagai fungsi

In [ ]:
def mean_intra_distance(X, labels):
    """Rata-rata jarak antar pasangan titik dalam kelompok yang sama, per kelompok."""
    result = {}
    for k in np.unique(labels):
        members = X[labels == k]
        d = pairwise_distances(members)
        n = len(members)
        result[k] = d.sum() / (n * (n - 1))   # tiap pasangan terhitung dua kali, diagonal nol
    return result


def cohesion(X, labels):
    """Rata-rata jarak titik ke centroid kelompoknya, per kelompok."""
    result = {}
    for k in np.unique(labels):
        members = X[labels == k]
        centroid = members.mean(axis=0)
        result[k] = np.linalg.norm(members - centroid, axis=1).mean()
    return result


def centroid_separation(X, labels):
    """Jarak antar centroid untuk setiap pasangan kelompok."""
    ks = np.unique(labels)
    centroids = np.array([X[labels == k].mean(axis=0) for k in ks])
    d = pairwise_distances(centroids)
    return {(ks[i], ks[j]): d[i, j] for i in range(len(ks)) for j in range(i + 1, len(ks))}


def min_inter_distance(X, labels):
    """Jarak terkecil antara dua titik dari kelompok berbeda, untuk setiap pasangan."""
    ks = np.unique(labels)
    result = {}
    for i in range(len(ks)):
        for j in range(i + 1, len(ks)):
            d = pairwise_distances(X[labels == ks[i]], X[labels == ks[j]])
            result[(ks[i], ks[j])] = d.min()
    return result

## Bagian D: Menghitung ukuran pada data kecil

In [ ]:
print("Centroid kelompok 0:", X_tiny[labels_tiny == 0].mean(axis=0).round(3))
print("Centroid kelompok 1:", X_tiny[labels_tiny == 1].mean(axis=0).round(3))
print("Rata-rata jarak dalam kelompok :", {int(k): round(float(v), 3) for k, v in mean_intra_distance(X_tiny, labels_tiny).items()})
print("Cohesion (jarak ke centroid)   :", {int(k): round(float(v), 3) for k, v in cohesion(X_tiny, labels_tiny).items()})
print("Jarak antar centroid           :", {(int(a), int(b)): round(float(v), 3) for (a, b), v in centroid_separation(X_tiny, labels_tiny).items()})
print("Jarak antar titik terdekat     :", {(int(a), int(b)): round(float(v), 3) for (a, b), v in min_inter_distance(X_tiny, labels_tiny).items()})

## Bagian E: Kelompok dekat dan kelompok jauh

In [ ]:
# Dua dataset dengan sebaran sama (std = 1.0), hanya jarak antar pusat yang berbeda
centers_near = [[0, 0], [3, 0]]
centers_far = [[0, 0], [10, 0]]
X_near, _ = make_blobs(n_samples=200, centers=centers_near, cluster_std=1.0, random_state=42)
X_far, _ = make_blobs(n_samples=200, centers=centers_far, cluster_std=1.0, random_state=42)

pair_data = {"Dekat (pusat 3 satuan)": X_near, "Jauh (pusat 10 satuan)": X_far}
pair_labels = {}
rows = []
for name, X in pair_data.items():
    labels = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X)
    pair_labels[name] = labels
    coh = np.mean(list(cohesion(X, labels).values()))
    sep = list(centroid_separation(X, labels).values())[0]
    rows.append(
        {
            "Dataset": name,
            "Cohesion rata-rata": coh,
            "Jarak antar centroid": sep,
            "Rasio separation/cohesion": sep / coh,
            "Silhouette": silhouette_score(X, labels),
        }
    )
print(pd.DataFrame(rows).set_index("Dataset").round(3).to_string())

## Bagian F: Menggambar kelompok dekat dan jauh

In [ ]:
palette = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, (name, X) in zip(axes, pair_data.items()):
    labels = pair_labels[name]
    for k in range(2):
        ax.scatter(X[labels == k, 0], X[labels == k, 1], s=14, color=palette[k])
        c = X[labels == k].mean(axis=0)
        ax.scatter(c[0], c[1], s=180, marker="X", color="black")
    ax.set_title(name)
    ax.set_xlabel("Fitur 1")
axes[0].set_ylabel("Fitur 2")
plt.tight_layout()
plt.show()

## Bagian G: Kepadatan dua kelompok

In [ ]:
# Dua kelompok dengan kepadatan berbeda: kelompok 0 rapat, kelompok 1 renggang
X_dens, y_dens = make_blobs(
    n_samples=[150, 150], centers=[[0, 0], [8, 0]], cluster_std=[0.4, 1.6], random_state=42
)
for k in (0, 1):
    members = X_dens[y_dens == k]
    centroid = members.mean(axis=0)
    coh = np.linalg.norm(members - centroid, axis=1).mean()
    # n_neighbors=2: tetangga pertama adalah titik itu sendiri, jadi ambil kolom kedua
    nn = NearestNeighbors(n_neighbors=2).fit(members)
    dist, _ = nn.kneighbors(members)
    print(f"Kelompok {k}: cohesion = {coh:.3f}, rata-rata jarak ke tetangga terdekat = {dist[:, 1].mean():.3f}")

## Bagian H: Keterbatasan: label benar bisa kalah skor

In [ ]:
# Keterbatasan: pada data moons, label BENAR bisa mendapat skor internal lebih rendah
X_moons, y_moons = make_moons(n_samples=300, noise=0.06, random_state=42)
labels_km = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X_moons)

print(f"Silhouette label K-Means : {silhouette_score(X_moons, labels_km):.3f}")
print(f"Silhouette label benar   : {silhouette_score(X_moons, y_moons):.3f}")
print("Cohesion label K-Means   :", round(float(np.mean(list(cohesion(X_moons, labels_km).values()))), 3))
print("Cohesion label benar     :", round(float(np.mean(list(cohesion(X_moons, y_moons).values()))), 3))

## Bagian I: Menggambar data moons

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
for ax, labels, title in zip(
    axes,
    [labels_km, y_moons],
    [f"Hasil K-Means\nSilhouette = {silhouette_score(X_moons, labels_km):.3f}",
     f"Label benar\nSilhouette = {silhouette_score(X_moons, y_moons):.3f}"],
):
    for k in range(2):
        ax.scatter(X_moons[labels == k, 0], X_moons[labels == k, 1], s=14, color=palette[k])
    ax.set_title(title)
    ax.set_xlabel("Fitur 1")
axes[0].set_ylabel("Fitur 2")
plt.tight_layout()
plt.show()

## Bagian J: Ukuran dasar pada Mall Customers

In [ ]:
df = pd.read_csv("data/mall_customers.csv")
features = ["Annual Income (k$)", "Spending Score (1-100)"]
X_mall = StandardScaler().fit_transform(df[features])
labels_mall = KMeans(n_clusters=5, n_init=10, random_state=42).fit_predict(X_mall)

coh_mall = cohesion(X_mall, labels_mall)
sizes = np.bincount(labels_mall)
table = pd.DataFrame({"Ukuran": sizes, "Cohesion": [coh_mall[k] for k in range(5)]})
print(table.round(3))
sep_mall = centroid_separation(X_mall, labels_mall)
nearest_pair = min(sep_mall, key=sep_mall.get)
farthest_pair = max(sep_mall, key=sep_mall.get)
print("Pasangan kelompok dengan centroid terdekat:", (int(nearest_pair[0]), int(nearest_pair[1])), round(float(sep_mall[nearest_pair]), 3))
print("Pasangan kelompok dengan centroid terjauh :", (int(farthest_pair[0]), int(farthest_pair[1])), round(float(sep_mall[farthest_pair]), 3))

## Latihan

1. Hitung manual cohesion kelompok 1 pada `X_tiny` dan cocokkan dengan output.
2. Pada `X_tiny`, ganti titik \( (6,6) \) menjadi \( (9,9) \). Apa cohesion tiap kelompok, jarak antar centroid, dan jarak antar titik terdekat sekarang?
3. Pada eksperimen dekat vs jauh, ganti `cluster_std=1.0` pada dataset "dekat" menjadi `2.0`. Berapa Silhouette dan cohesion rata-ratanya?
4. Jalankan K-Means dengan K = 3 pada Mall Customers (terskala). Hitung cohesion rata-rata berbobot ukuran kelompok dan bandingkan dengan K = 5.
5. Mengapa cohesion yang sama pada dua dataset bisa menghasilkan Silhouette yang berbeda?
6. Mengapa pada moons, pembagian yang salah bisa mendapat skor internal lebih baik?

Jawaban lengkap ada di halaman bab.

## Ringkasan

- **Evaluasi internal** menilai hasil clustering hanya dari data dan label hasil clustering, tanpa label benar.
- Dua gagasan inti: **jarak dalam** (kecil lebih baik) dan **jarak antar** (besar lebih baik).
- Empat ukuran dasar: rata-rata jarak dalam kelompok, **cohesion** (jarak ke centroid), **separation** (jarak antar centroid dan antar titik terdekat), serta **density** (jarak ke tetangga terdekat).
- Pada enam titik kita menghitung manual dan mendapat angka yang sama dengan kode: cohesion 0,654, jarak antar centroid 6,403, jarak antar titik terdekat 5,657.
- Pada sebaran yang sama, kelompok yang lebih berjauhan mendapat Silhouette lebih tinggi (0,525 vs 0,834).
- Evaluasi internal punya asumsi **kompak dan terpisah**. Pada moons, label benar mendapat Silhouette 0,330, lebih rendah daripada K-Means yang salah bentuk (0,486).
- Samakan skala fitur dan lihat hasil **per kelompok**, bukan hanya rata-rata.